In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from types import SimpleNamespace
from scipy.fft import fftn, ifftn  # faster than np.fft on CPU

from shift import Shift
from utils import *

# Acquisition parameters

In [ ]:
n = 64
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817]) * 1e-3 - sx0
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]*0+1
nobj = n#int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 


# Modeling data

In [ ]:
cl = Shift(n,nobj,n,nobj,1/norm_magnifications,'complex64')

a = (cp.random.random([1,nobj,nobj])+1j*cp.random.random([1,nobj,nobj])).astype('complex64')
b = (cp.random.random([1,n,n])+1j*cp.random.random([1,n,n])).astype('complex64')
pos = (cp.random.random([1,4,2])).astype('float32')
bb=cl.curlyS(a,pos,3)


In [ ]:
pad=n//4
a = cp.zeros([1,nobj,nobj]).astype('complex64')
a[:,pad:-pad,pad:-pad] = (-1+1j/100)/8
pos[:]=0
pos[...,1]=0.25
b = cl.curlyS(a,pos,3)
# c = cl.curlySback(a,-pos,1)


In [ ]:
mshow_complex(a[0],True)
mshow_complex(b[0],True)
plt.plot(a[0,n//2].real.get())
plt.plot(b[0,n//2].real.get())
# plt.plot(c[0,64].real.get())


# Comparison with cubic interpolation (scipy / cupyx)

In [ ]:
# The kernel maps: output[ty,tx] = input[ty - ry, tx - rx]  (pixel units)
# pos shape [ntheta, ndist, 2]: r[0]=y-shift, r[1]=x-shift, read as r[2*tz], r[2*tz+1]
# With ntheta=1, tz=0 → uses pos[0,0,:] as the shift vector
ry = float(pos[0, 0, 0].get())
rx = float(pos[0, 0, 1].get())
print(f"shift: ry={ry}, rx={rx} pixels")

a_np = a[0].get()   # [nobj, nobj] complex64

# --- scipy.ndimage (CPU, cubic B-spline order=3) ---
b_scipy_re = sp.ndimage.shift(a_np.real, [ry, rx], order=3, mode='constant', cval=0)
b_scipy_im = sp.ndimage.shift(a_np.imag, [ry, rx], order=3, mode='constant', cval=0)
b_scipy = (b_scipy_re + 1j * b_scipy_im).astype('complex64')

# --- cupyx.scipy.ndimage (GPU, cubic B-spline order=3) ---
import cupyx.scipy.ndimage as cpnd
a_cp = a[0]
b_cupyx_re = cpnd.shift(a_cp.real, [ry, rx], order=3, mode='constant', cval=0)
b_cupyx_im = cpnd.shift(a_cp.imag, [ry, rx], order=3, mode='constant', cval=0)
b_cupyx = (b_cupyx_re + 1j * b_cupyx_im).astype('complex64')

b_bs = b[0].get()   # curlyS result

print(f"max |curlyS - scipy|  = {np.max(np.abs(b_bs - b_scipy)):.3e}")
print(f"max |curlyS - cupyx|  = {np.max(np.abs(b_bs - b_cupyx.get())):.3e}")
print(f"max |scipy  - cupyx|  = {np.max(np.abs(b_scipy - b_cupyx.get())):.3e}")

In [ ]:
# Visual comparison along the central row
row = nobj // 2
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
for ax, part, attr in zip(axs, ['real', 'imag'], ['real', 'imag']):
    ax.plot(getattr(b_bs,       attr)[row], label='curlyS (B-spline)',  lw=2)
    ax.plot(getattr(b_scipy,    attr)[row], '-', label='scipy cubic',  lw=1.5)
    # ax.plot(getattr(b_cupyx.get(), attr)[row], ':', label='cupyx cubic', lw=1.5)
    ax.set_title(part)
    ax.legend()
    ax.grid()
plt.suptitle(f"Central row comparison — shift (ry={ry}, rx={rx} px)")
plt.tight_layout()
plt.show()

# Difference images
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
diff = np.abs(b_bs - b_scipy)
im = axs[0].imshow(diff, cmap='hot')
axs[0].set_title('|curlyS − scipy cubic|')
fig.colorbar(im, ax=axs[0])
diff2 = np.abs(b_bs - b_cupyx.get())
im = axs[1].imshow(diff2, cmap='hot')
axs[1].set_title('|curlyS − cupyx cubic|')
fig.colorbar(im, ax=axs[1])
plt.tight_layout()
plt.show()

In [ ]:
from scipy.interpolate import PchipInterpolator, Akima1DInterpolator


def _shift_1d(row, shift, interp_cls):
    """Shift a real 1D array by `shift` pixels; zero-pad outside bounds."""
    n = len(row)
    x = np.arange(n, dtype=float)
    out = interp_cls(x, row)(x - shift)   # output[j] = input[j - shift]
    return np.nan_to_num(out, nan=0.0).astype(row.dtype)


def _shift_2d_real(arr, ry, rx, interp_cls):
    """Separable 2D shift: rows first (x), then columns (y)."""
    tmp = np.apply_along_axis(_shift_1d, 1, arr,  rx, interp_cls) if rx else arr.copy()
    out = np.apply_along_axis(_shift_1d, 0, tmp,  ry, interp_cls) if ry else tmp
    return out


def shift_2d(arr_c, ry, rx, interp_cls):
    re = _shift_2d_real(arr_c.real, ry, rx, interp_cls)
    im = _shift_2d_real(arr_c.imag, ry, rx, interp_cls)
    return (re + 1j * im).astype(arr_c.dtype)


b_pchip = shift_2d(a_np, ry, rx, PchipInterpolator)
b_akima = shift_2d(a_np, ry, rx, Akima1DInterpolator)

print(f"max |curlyS - pchip|  = {np.max(np.abs(b_bs - b_pchip)):.3e}")
print(f"max |curlyS - akima|  = {np.max(np.abs(b_bs - b_akima)):.3e}")
print(f"max |scipy  - pchip|  = {np.max(np.abs(b_scipy - b_pchip)):.3e}")
print(f"max |scipy  - akima|  = {np.max(np.abs(b_scipy - b_akima)):.3e}")

In [ ]:
methods = {
    'curlyS (B-spline)': b_bs,
    'scipy cubic':       b_scipy,
    'cupyx cubic':       b_cupyx.get(),
    'PCHIP':             b_pchip,
    'Akima':             b_akima,
}

# --- Line plot along central row ---
row = nobj // 2
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
styles = ['-', '--', ':', '-.', (0, (3, 1, 1, 1))]
for ax, attr in zip(axs, ['real', 'imag']):
    for (label, arr), ls in zip(methods.items(), styles):
        ax.plot(getattr(arr, attr)[row], ls=ls, label=label, lw=1.8)
    ax.set_title(attr)
    ax.legend(fontsize=8)
    ax.grid()
plt.suptitle(f"Central row — shift (ry={ry}, rx={rx} px)")
plt.tight_layout()
plt.show()

# --- Difference images vs curlyS ---
ref = b_bs
others = {k: v for k, v in methods.items() if k != 'curlyS (B-spline)'}
fig, axs = plt.subplots(1, len(others), figsize=(4 * len(others), 4))
for ax, (label, arr) in zip(axs, others.items()):
    im = ax.imshow(np.abs(ref - arr), cmap='hot')
    ax.set_title(f'|curlyS − {label}|')
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Comparison on test shapes (discontinuity, monotone ramp, smooth Gaussian)

In [ ]:
from scipy.interpolate import PchipInterpolator, Akima1DInterpolator

# --- 1D test profiles --------------------------------------------------
cx  = n // 2
xp  = np.arange(n, dtype='float64')

test_shapes = {
    'Box (discontinuity)': np.where(np.abs(xp - cx) < n // 6, 1.0, 0.0),
    'Ramp (monotone)':     np.clip((xp - (cx - n//6)) / (n // 3), 0.0, 1.0),
    'Gaussian (smooth)':   np.exp(-((xp - cx) ** 2) / (2 * (n // 8) ** 2)),
}


# --- helpers -----------------------------------------------------------
def _shift_scipy(arr2d, ry, rx, order):
    """scipy.ndimage.shift applied to real part only (objects are real-valued)."""
    return sp.ndimage.shift(arr2d.real, [ry, rx], order=order,
                            mode='constant', cval=0).astype('float32')


# --- run all methods for each shape -----------------------------------
METHOD_STYLES = {
    'curlyS (B-spline)': ('-',  'C0', 2.2),
    'linear':            (':',  'C4', 1.8),
    'scipy cubic':       ('--', 'C1', 1.8),
    'PCHIP':             ('-.',  'C2', 1.8),
    'Akima':             ((0,(4,1,1,1)), 'C3', 1.8),
}

results = {}
for name, prof in test_shapes.items():
    arr2d  = np.tile(prof, (n, 1)).astype('complex64')   # [n, n], imag=0
    arr_gpu = cp.array(arr2d[np.newaxis])                # [1, n, n]

    results[name] = {
        'input':            prof.astype('float32'),
        'curlyS (B-spline)': cl.curlyS(arr_gpu, pos, 3)[0].get().real,
        'linear':            _shift_scipy(arr2d, ry, rx, order=1),
        'scipy cubic':       _shift_scipy(arr2d, ry, rx, order=3),
        'PCHIP':             shift_2d(arr2d, ry, rx, PchipInterpolator).real,
        'Akima':             shift_2d(arr2d, ry, rx, Akima1DInterpolator).real,
    }

print("done")

In [ ]:
row = n // 2

fig, axs = plt.subplots(len(test_shapes), 2, figsize=(14, 4 * len(test_shapes)))

for i, (name, res) in enumerate(results.items()):
    prof_row = res['input']

    # zoom: centre on the pixel of steepest gradient
    peak_idx  = int(np.argmax(np.abs(np.gradient(prof_row))))
    half_zoom = 14
    z = slice(max(0, peak_idx - half_zoom), min(n, peak_idx + half_zoom + 1))

    for j, (ax, xdata, title_suffix) in enumerate([
        (axs[i, 0], xp,       'full view'),
        (axs[i, 1], xp[z],    'zoom near transition'),
    ]):
        ax.plot(xdata, prof_row[z if j else slice(None)],
                'k', lw=1.2, ls='--', alpha=0.45, label='input (unshifted)')
        for key, (ls, color, lw) in METHOD_STYLES.items():
            vals = res[key][row][z if j else slice(None)]
            ax.plot(xdata, vals, ls=ls, color=color, lw=lw, label=key)
        ax.set_title(f'{name} — {title_suffix}')
        ax.legend(fontsize=7.5, ncol=2)
        ax.grid()

plt.suptitle(f'Shift (ry={ry}, rx={rx} px) — real part', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()